# Convergence of a single point vortex field

In the [solid-body rotation notebook](./solid-body-rotation-2d.ipynb), the velocity field was linear, so Parcels' barycentric interpolation was able to reproduce it exactly and the particle trajectories converged with `dt` all the way down to floating-point round-off. Here we use a flow in which the linear interpolation is not exact.

The particles still move on exact circles, but the velocity now falls off as $1/r$. Within each triangle of the grid, linear interpolation can only approximate this, and we will see that the resulting interpolation error stops the convergence long before we reach round-off.

In [ ]:
import os

import numpy as np
import parcels
import uxarray as ux
from IPython.display import HTML
from matplotlib import pyplot as plt
from matplotlib import animation, tri
from parcels._datasets.unstructured.generated import solid_body_rotation_node_centered

## The point vortex field

The velocity of a point vortex with circulation $\Gamma$ at the origin is

$$
u = -\frac{\Gamma y}{2\pi r^2}, \qquad v = \frac{\Gamma x}{2\pi r^2}, \qquad r^2 = x^2 + y^2.
$$

The speed $\Gamma / (2\pi r)$ decreases with distance from the vortex, so particles closer to the centre rotate faster. A particle at radius $R_0$ has angular speed and rotation period

$$
\omega(R_0) = \frac{\Gamma}{2\pi R_0^2}, \qquad T(R_0) = \frac{4\pi^2 R_0^2}{\Gamma}.
$$

Similarly to the solid-body rotation case, we use the parcels build in `solid_body_rotation_node_centered` function to generate the grid, except this time we overwrite `U` and `V` with the vortex velocity at the nodes. We also use a finer grid so that the linear interpolation error doesn't instantly overwhelm the time stepping error. The lattice has an even number of points (160) per side, so no node sits at the origin, where the velocity would be undefined as 1/0. We choose $\Gamma$ so that the innermost particle ($R_0 = 3$) completes three rotations over a 12 hour runtime.

In [ ]:
ds = solid_body_rotation_node_centered(nx=120)
x, y = ds.uxgrid.node_lon.values, ds.uxgrid.node_lat.values
r2 = x**2 + y**2
R0 = np.linspace(3.0, 4.5, 10)
runtime_s = 43200
# Set so the fastest (innermost) particle makes three rotations over the runtime
gamma = 3 * 4 * np.pi**2 * R0.min() ** 2 / runtime_s
ds["U"].values[:] = -gamma * y / (2 * np.pi * r2)
ds["V"].values[:] = gamma * x / (2 * np.pi * r2)

In [ ]:
fieldset = parcels.FieldSet.from_ugrid_conventions(ds, mesh="flat")

In [ ]:
fieldset.U.interp_method

Note the Parcels default interpolator as UxLinearNodeConstantZC(), which does barycentric linear interpolation in the horizontal direction and piecewise constant interpolation in the vertical.

## Release particles

We release 10 particles along the positive $x$-axis ($\theta_0 = 0$), at radii $R_0$ evenly spaced between 3.0 and 4.5. We keep the particles away from the vortex centre so that they don't orbit too fast.

In [ ]:
theta0 = 0.0
pset = parcels.ParticleSet(fieldset, x=R0 * np.cos(theta0), y=R0 * np.sin(theta0))

## Analytical solution

Each particle moves on a circle at its own angular speed $\omega(R_0)$:

$$
x(t) = R_0 \cos\big(\omega(R_0)\, t + \theta_0\big), \qquad y(t) = R_0 \sin\big(\omega(R_0)\, t + \theta_0\big).
$$

We evaluate this every 60 s over the 12 hour runtime.

In [ ]:
omega = gamma / (2 * np.pi * R0**2)
t = np.arange(0, runtime_s + 1, 60)
x_analytical = R0 * np.cos(omega * t[:, None] + theta0)
y_analytical = R0 * np.sin(omega * t[:, None] + theta0)

We animate the particles moving along their analytical trajectories on top of the grid (faint black lines), with frames 10 minutes apart.

In [ ]:
plt.figure(figsize=(8, 8))
grid = ds.uxgrid
if grid is not None:
    plt.triplot(tri.Triangulation(grid.node_lon.values, grid.node_lat.values, grid.face_node_connectivity.values), color="k", alpha=0.2, linewidth=0.7)
lines = plt.plot(x_analytical, y_analytical)
dots = plt.scatter(pset.x, pset.y, c=[line.get_color() for line in lines], s=10)
anim = animation.FuncAnimation(plt.gcf(), lambda i: dots.set_offsets(np.column_stack([x_analytical[i], y_analytical[i]])), frames=range(0, len(t), 30))
plt.close()
HTML(anim.to_jshtml())

## Convergence with RK2

We run the same RK2 convergence using six timesteps, halving `dt` from 960 s to 30 s. The error of each particle is again the distance between its final position and the analytical final position.

In [ ]:
os.makedirs("parquet", exist_ok=True)

In [ ]:
dts, errors = [920, 480, 240, 120, 60, 30], []
for dt_s in dts:
    pset = parcels.ParticleSet(fieldset, x=R0 * np.cos(theta0), y=R0 * np.sin(theta0))
    output_file = parcels.ParticleFile(f"parquet/single-point-vortex-dt{dt_s}.parquet", outputdt=np.timedelta64(960, "s"), mode="w")
    pset.execute(parcels.kernels.AdvectionRK2, runtime=np.timedelta64(runtime_s, "s"), dt=np.timedelta64(dt_s, "s"), output_file=output_file)
    errors.append(np.hypot(pset.x - x_analytical[-1], pset.y - y_analytical[-1]))

In [ ]:
plt.semilogy(dts, errors)
plt.semilogy(dts, 2 * errors[-1].max() * (np.array(dts) / dts[-1]) ** 2, "k--", label=r"$\propto dt^2$")
plt.legend()
plt.xlabel("dt [s]")
plt.ylabel("error")
plt.show()

At the largest timesteps the time-stepping error dominates, and the errors fall as `dt` decreases. From around `dt` = 240 s, however, the expected RK2 convergence breaks down. The resulting error is roughly $10^{-3}$ even for a `dt` of 30. This is significantly above `float32` round-off level of around $10^{-6}$ reached in the solid-body case. This plateau is the interpolation error that comes from using a linear interpolator on a nonlinear velocity field.

## Refining the grid

The error of linear interpolation shrinks as the grid spacing $h$ decreases, so a finer grid should allow for more accurate particle simulations. We generate the same vortex field on an 240 x 240 lattice, which roughly halves $h$, and repeat the RK2 runs.

In [ ]:
ds_fine = solid_body_rotation_node_centered(nx=240)
x_fine, y_fine = ds_fine.uxgrid.node_lon.values, ds_fine.uxgrid.node_lat.values
ds_fine["U"].values[:] = -gamma * y_fine / (2 * np.pi * (x_fine**2 + y_fine**2))
ds_fine["V"].values[:] = gamma * x_fine / (2 * np.pi * (x_fine**2 + y_fine**2))
fieldset_fine = parcels.FieldSet.from_ugrid_conventions(ds_fine, mesh="flat")

In [ ]:
errors_fine = []
for dt_s in dts:
    pset = parcels.ParticleSet(fieldset_fine, x=R0 * np.cos(theta0), y=R0 * np.sin(theta0))
    output_file = parcels.ParticleFile(f"parquet/single-point-vortex-fine-dt{dt_s}.parquet", outputdt=np.timedelta64(960, "s"), mode="w")
    pset.execute(parcels.kernels.AdvectionRK2, runtime=np.timedelta64(runtime_s, "s"), dt=np.timedelta64(dt_s, "s"), output_file=output_file)
    errors_fine.append(np.hypot(pset.x - x_analytical[-1], pset.y - y_analytical[-1]))

In [ ]:
plt.semilogy(dts, errors_fine)
plt.semilogy(dts, 2 * errors_fine[-1].max() * (np.array(dts) / dts[-1]) ** 2, "k--", label=r"$\propto dt^2$")
plt.legend()
plt.xlabel("dt [s]")
plt.ylabel("error")
plt.show()

On the finer grid the plateau is lower, so the errors keep decreasing to somewhat smaller `dt` before they level off. Refining the grid reduces the interpolation error, but does not remove it. Getting close to round-off this way would require ever finer grids, which quickly become expensive. Even after doubling the resolution here we don't see the RK2 scaling hold for small `dt`.

## A custom interpolator for the $1/r$ field

Instead of refining the grid, we can use what we know about the field. Multiplying the velocity by $r^2$ gives

$$
u\, r^2 = -\frac{\Gamma y}{2\pi}, \qquad v\, r^2 = \frac{\Gamma x}{2\pi},
$$

which is linear in $x$ and $y$. So we can interpolate $u\, r^2$ barycentrically and then divide by the particle's own $r^2$:

$$
u(\mathbf{x}_p) = \frac{1}{r_p^2} \sum_{i=1}^{3} \lambda_i\, u_i\, r_i^2,
$$

where $\lambda_i$ are the barycentric coordinates of the particle in its triangle, and $u_i$ and $r_i$ are the velocity and radius at the triangle's three nodes (and likewise for $v$).

In [ ]:
class UxR2WeightedNodeConstantZC(parcels.interpolators.UxLinearNodeConstantZC):
    def interp(self, particle_positions, grid_positions, field):
        node_ids = field.grid.uxgrid.face_node_connectivity[grid_positions["FACE"]["index"], :].values
        node_r2 = field.grid.uxgrid.node_lon.values[node_ids] ** 2 + field.grid.uxgrid.node_lat.values[node_ids] ** 2
        r2_weighted = (field.data.values[0, 0][node_ids] * node_r2 * grid_positions["FACE"]["bcoord"]).sum(axis=1)
        return r2_weighted / (particle_positions["x"] ** 2 + particle_positions["y"] ** 2)

We assign the new interpolator to both `U` and `V` by setting their `interp_method`, and repeat the RK2 runs on the original 120 × 120 grid.

In [ ]:
fieldset.U.interp_method = fieldset.V.interp_method = UxR2WeightedNodeConstantZC()
errors_r2 = []
for dt_s in dts:
    pset = parcels.ParticleSet(fieldset, x=R0 * np.cos(theta0), y=R0 * np.sin(theta0))
    output_file = parcels.ParticleFile(f"parquet/single-point-vortex-r2-dt{dt_s}.parquet", outputdt=np.timedelta64(960, "s"), mode="w")
    pset.execute(parcels.kernels.AdvectionRK2, runtime=np.timedelta64(runtime_s, "s"), dt=np.timedelta64(dt_s, "s"), output_file=output_file)
    errors_r2.append(np.hypot(pset.x - x_analytical[-1], pset.y - y_analytical[-1]))

We plot the errors with the custom interpolator in the same way.

In [ ]:
plt.semilogy(dts, errors_r2)
plt.semilogy(dts, 2 * errors_r2[-1].max() * (np.array(dts) / dts[-1]) ** 2, "k--", label=r"$\propto dt^2$")
plt.legend()
plt.xlabel("dt [s]")
plt.ylabel("error")
plt.show()

With the exact interpolator the plateau disappears and the errors of all particles decrease parallel to the $\text{d}t^2$ line over the whole range of timesteps, as in solid-body rotation. At `dt` = 30 s they are still well above `float32` round-off, so smaller timesteps would keep improving the result.